 ## Duplicate Handling

### Why does this matter?
Duplicates are one of the most common data quality issues in real datasets. When you merge two tables, a row might appear twice. When data is collected from multiple sources, the same record gets entered twice. When a form is submitted twice. When a CSV is appended to itself by mistake. Duplicates silently corrupt your analysis — they inflate counts, skew averages, and make totals wrong. Finding and removing them is a non-negotiable step in every data cleaning pipeline.

### What is a duplicate?
What is it?
A duplicate row is a row where the values in one or more columns are identical to another row. It can be a full duplicate — every single column matches — or a partial duplicate — only specific key columns match (like same customer ID appearing twice).

### Why does partial duplicate matter more in real work?
Because full duplicates are rare. More commonly, the same entity appears twice with slightly different data — same customer ID but different timestamps, same order ID but different status. These are partial duplicates based on a key column, and they're more dangerous because they're harder to spot.

### How does Pandas detect duplicates?
Pandas compares rows element by element. For full duplicates it checks all columns. For partial duplicates you specify which columns to check using the *subset* parameter.

In [1]:
import pandas as pd

data = {
    'EmpID':  [101, 102, 103, 101, 104, 102, 105],
    'Name':   ['Sumed','Priya','Rahul','Sumed','Neha','Priya','Arjun'],
    'Dept':   ['Data','HR','Data','Data','Finance','HR','Data'],
    'Salary': [60000,45000,75000,60000,55000,48000,80000]
}
# EmpID 101 (Sumed) appears at index 0 and 3 — full duplicate
# EmpID 102 (Priya) appears at index 1 and 5 — partial duplicate
# (same EmpID and Name but different Salary)
df = pd.DataFrame(data)

In [2]:
df

,EmpID,Name,Dept,Salary
0,101,Sumed,Data,60000
1,102,Priya,HR,45000
2,103,Rahul,Data,75000
3,101,Sumed,Data,60000
4,104,Neha,Finance,55000
5,102,Priya,HR,48000
6,105,Arjun,Data,80000


*Red rows are duplicates. Row 3 = full duplicate of row 0 (every column matches). Row 5 = partial duplicate of row 1 (same EmpID and Name but different Salary — data entry error).*

## Duplicated(): detect duplicates

In [4]:
# duplicated() → returns a boolean Series
# True = this row is a duplicate of an earlier row
# False = this row is unique (or the FIRST occurrence)

df.duplicated()
# By default checks ALL columns
# keep='first' (default) → first occurrence = False, rest = True
# keep='last'  → last occurrence = False, earlier ones = True
# keep=False   → ALL occurrences of duplicates = True


0    False
1    False
2    False
3     True
4    False
5    False
6    False
dtype: bool

In [6]:

# How many duplicates total?
df.duplicated().sum()       # → 1 (only row 3 is full duplicate)



np.int64(1)

In [7]:
# VIEW the duplicate rows
df[df.duplicated()]          # shows only the duplicate rows
df[df.duplicated(keep=False)] # shows ALL rows involved in duplicates

,EmpID,Name,Dept,Salary
0,101,Sumed,Data,60000
3,101,Sumed,Data,60000


*Row 5 (Priya, 48000) is NOT flagged as a full duplicate because Salary differs. This is why checking ALL columns isn't always enough — you need subset to catch partial duplicates.*

## Subset(): detecting partial duplicates

In [9]:
# subset → check only specific columns for duplicates
# Use when you care about key columns, not ALL columns
# Real use: same EmpID should never appear twice

df.duplicated(subset=['EmpID'])
# Now row 5 (Priya, 48000) IS flagged — same EmpID as row 1


0    False
1    False
2    False
3     True
4    False
5     True
6    False
dtype: bool

In [10]:

# Count partial duplicates on EmpID
df.duplicated(subset=['EmpID']).sum()    # → 2



np.int64(2)

In [11]:
# View ALL rows involved in EmpID duplicates
df[df.duplicated(subset=['EmpID'], keep=False)]


,EmpID,Name,Dept,Salary
0,101,Sumed,Data,60000
1,102,Priya,HR,45000
3,101,Sumed,Data,60000
5,102,Priya,HR,48000


In [12]:

# Multiple subset columns — duplicate if BOTH match
df.duplicated(subset=['Name', 'Dept'])
# Flag row only if Name AND Dept are both the same

0    False
1    False
2    False
3     True
4    False
5     True
6    False
dtype: bool

*keep=False is your investigation tool — it shows every row involved in a duplicate relationship so you can decide which to keep and which to drop.*

## Drop_duplicates(): remove duplocates

In [13]:
# drop_duplicates() → removes duplicate rows
# Returns a new DataFrame — original unchanged

# Remove full duplicates — keep first occurrence
df.drop_duplicates()

# keep='first' (default) → keep first occurrence, drop rest
# keep='last'  → keep last occurrence, drop earlier ones
# keep=False   → drop ALL occurrences of duplicates

df.drop_duplicates(keep='first')   # keeps row 0, drops row 3
df.drop_duplicates(keep='last')    # keeps row 3, drops row 0
df.drop_duplicates(keep=False)    # drops BOTH row 0 and row 3

# subset → remove partial duplicates on key columns
df.drop_duplicates(subset=['EmpID'], keep='first')
# Keeps first occurrence of each EmpID → row 1 kept, row 5 dropped

# Always reassign to save the result
df = df.drop_duplicates(subset=['EmpID'], keep='first').reset_index(drop=True)

In [14]:
df

,EmpID,Name,Dept,Salary
0,101,Sumed,Data,60000
1,102,Priya,HR,45000
2,103,Rahul,Data,75000
3,104,Neha,Finance,55000
4,105,Arjun,Data,80000


*Always chain .reset_index(drop=True) after drop_duplicates — index will have gaps (0,1,2,4,6) after removal. Reset gives clean 0,1,2,3,4.*

## KEEP='LAST'  : when last record is most recent

In [15]:
# Real world scenario: transaction log with updates
# Same OrderID appears multiple times as status updates
# You want the LATEST status → keep='last'

orders = pd.DataFrame({
    'OrderID':  [1001, 1002, 1001, 1003, 1002],
    'Status':   ['Placed','Placed','Shipped','Placed','Delivered'],
    'UpdatedAt':['2024-01-01','2024-01-02','2024-01-05',
                 '2024-01-03','2024-01-08']
})

# Sort by date first — ensures last = most recent
orders = orders.sort_values('UpdatedAt')

# Then keep last → most recent record per OrderID
latest = orders.drop_duplicates(subset=['OrderID'], keep='last')
latest = latest.reset_index(drop=True)
print(latest)

   OrderID     Status   UpdatedAt
0     1003     Placed  2024-01-03
1     1001    Shipped  2024-01-05
2     1002  Delivered  2024-01-08


*sort_values() → drop_duplicates(keep='last') is one of the most common real-world patterns. Always sort before deduplicating when order matters — otherwise "last" is meaningless.*

## Real world duplicate handling pipeline

In [16]:
# Professional duplicate handling workflow
# Never blindly drop — investigate first

# Step 1: How many full duplicates?
full_dups = df.duplicated().sum()
print(f"Full duplicates: {full_dups}")

# Step 2: How many partial duplicates on key column?
partial_dups = df.duplicated(subset=['EmpID']).sum()
print(f"Partial duplicates on EmpID: {partial_dups}")

# Step 3: Investigate — view all involved rows
df[df.duplicated(subset=['EmpID'], keep=False)]

# Step 4: Decide strategy and remove
# If full duplicates → keep first
# If partial (data update) → sort by date, keep last
# If partial (data conflict) → escalate to data owner
df_clean = df.drop_duplicates(subset=['EmpID'], keep='first')
df_clean = df_clean.reset_index(drop=True)

# Step 5: Verify — no duplicates remain
print(df_clean.duplicated(subset=['EmpID']).sum())  # → 0

Full duplicates: 0
Partial duplicates on EmpID: 0
0


*This 5-step workflow — count, investigate, decide strategy, remove, verify — is exactly what a senior analyst does. In interviews, describing this process shows you understand data quality, not just syntax.*

## Duplicates after merge: a common trap

In [17]:
# One of the most common sources of duplicates in real work
# When you merge two tables and the key is not unique in one of them

employees = pd.DataFrame({
    'EmpID': [1, 2, 3],
    'Name':  ['Sumed', 'Priya', 'Rahul']
})
trainings = pd.DataFrame({
    'EmpID':    [1, 1, 2],      # EmpID 1 attended TWO trainings
    'Training': ['Python', 'SQL', 'Excel']
})

result = pd.merge(employees, trainings, on='EmpID', how='left')
print(result)
# Sumed appears TWICE — once per training
# This is correct behavior but surprises beginners
# Always check shape after merge: result.shape vs employees.shape
print(f"Employees: {employees.shape}, Result: {result.shape}")

   EmpID   Name Training
0      1  Sumed   Python
1      1  Sumed      SQL
2      2  Priya    Excel
3      3  Rahul      NaN
Employees: (3, 2), Result: (4, 3)


*Always compare shape before and after merge. If result has MORE rows than the left table — your key is not unique in the right table. This is not always wrong (Sumed really did attend 2 trainings) but you must be aware of it.*